# Heart Disease Prediction: Exploratory Data Analysis & Preprocessing

This notebook performs comprehensive exploratory data analysis, data quality checks, missing value handling, categorical encoding, numerical normalization, and dataset preparation for predicting 10-year risk of Coronary Heart Disease (CHD) using the Framingham Heart Study dataset.

## 1. Environment Setup & Data Loading

In [ ]:
import matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
df = pd.read_csv('framingham.csv')

print("Dataset Shape:", df.shape)
print("\nData Types & Info")
df.info()

Dataset Shape: (4240, 16)

--- Data Types & Info ---
<class 'pandas.DataFrame'>
RangeIndex: 4240 entries, 0 to 4239
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4240 non-null   int64  
 1   age              4240 non-null   int64  
 2   education        4135 non-null   float64
 3   currentSmoker    4240 non-null   int64  
 4   cigsPerDay       4211 non-null   float64
 5   BPMeds           4187 non-null   float64
 6   prevalentStroke  4240 non-null   int64  
 7   prevalentHyp     4240 non-null   int64  
 8   diabetes         4240 non-null   int64  
 9   totChol          4190 non-null   float64
 10  sysBP            4240 non-null   float64
 11  diaBP            4240 non-null   float64
 12  BMI              4221 non-null   float64
 13  heartRate        4239 non-null   float64
 14  glucose          3852 non-null   float64
 15  TenYearCHD       4240 non-null   int64  
dtypes: float64(9), int

In [ ]:
df.head()

In [ ]:
df.describe().T

## 2. Target Variable & Data Quality Analysis

In [ ]:
# Calculating distribution of target variable TenYearCHD
target_counts = df['TenYearCHD'].value_counts()
target_pct = df['TenYearCHD'].value_counts(normalize=True) * 100

# Calculating target distribution percentages
print("Target Class Counts:\n", target_counts)
print("\nTarget Class Percentages (%):\n", target_pct.round(2))

Target Class Counts:
 TenYearCHD
0    3596
1     644
Name: count, dtype: int64

Target Class Percentages (%):
 TenYearCHD
0    84.81
1    15.19
Name: proportion, dtype: float64


In [ ]:
# Plotting target variable distribution

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(data=df, x='TenYearCHD', hue='TenYearCHD', palette='viridis', legend=False, ax=ax[0])
ax[0].set_title('TenYearCHD Count Distribution')
ax[0].set_xticks([0, 1])
ax[0].set_xticklabels(['No CHD (0)', 'CHD (1)'])

ax[1].pie(target_counts, labels=['No CHD (0)', 'CHD (1)'], autopct='%1.1f%%', colors=['#440154', '#21918c'], explode=(0, 0.1))
ax[1].set_title('TenYearCHD Percentage Share')
plt.tight_layout()
plt.close()

In [ ]:
# Audit missing values per column

missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)
print("Missing Values Breakdown:")
missing_summary

Missing Values Breakdown:


In [8]:
# Check for duplicate rows in dataset
print("Duplicate rows count:", df.duplicated().sum())

Duplicate rows count: 0


In [9]:
# Plot feature correlation matrix
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.close()

## 3. Handling Missing Values & Data Inconsistencies

In [10]:
# Copy dataset for clean processing
df_clean = df.copy()

# Fix inconsistency: non-smokers must have 0 cigsPerDay
df_clean.loc[df_clean['currentSmoker'] == 0, 'cigsPerDay'] = 0

# Impute categorical features using mode
for col in ['education', 'BPMeds']:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)

# Impute continuous numerical features using median
num_cols_missing = ['cigsPerDay', 'totChol', 'BMI', 'heartRate', 'glucose']
for col in num_cols_missing:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

# Verify missing values are resolved
print("Total missing values after imputation:", df_clean.isnull().sum().sum())

Total missing values after imputation: 0


## 4. Categorical Encoding & Numerical Feature Normalization

In [11]:
# Apply One-Hot Encoding to education feature
df_clean = pd.get_dummies(df_clean, columns=['education'], drop_first=True, dtype=int)

# Define continuous numerical features for scaling
continuous_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

# Normalize numerical continuous features with StandardScaler
scaler = StandardScaler()
df_clean[continuous_cols] = scaler.fit_transform(df_clean[continuous_cols])

# View preprocessed data sample
df_clean.head()

## 5. Input Features & Target Preparation

In [12]:
# Separate input feature matrix X and target vector y
X = df_clean.drop(columns=['TenYearCHD'])
y = df_clean['TenYearCHD']

# Print shapes of X and y
print("Feature matrix X shape:", X.shape)
print("Target vector y shape:", y.shape)

Feature matrix X shape: (4240, 17)
Target vector y shape: (4240,)


In [13]:
# Split data into train and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Display split sizes and class distributions
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nTrain target class proportion:\n", y_train.value_counts(normalize=True).round(4))
print("\nTest target class proportion:\n", y_test.value_counts(normalize=True).round(4))

X_train shape: (3392, 17)
X_test shape: (848, 17)

Train target class proportion:
 TenYearCHD
0    0.8482
1    0.1518
Name: proportion, dtype: float64

Test target class proportion:
 TenYearCHD
0    0.8479
1    0.1521
Name: proportion, dtype: float64


In [14]:
# Display prepared feature column names
print("Prepared Input Features:")
for i, feature in enumerate(X.columns, 1):
    print(f"{i}. {feature}")

Prepared Input Features:
1. male
2. age
3. currentSmoker
4. cigsPerDay
5. BPMeds
6. prevalentStroke
7. prevalentHyp
8. diabetes
9. totChol
10. sysBP
11. diaBP
12. BMI
13. heartRate
14. glucose
15. education_2.0
16. education_3.0
17. education_4.0
